# Chapter 19: Loop Engineering — The GAS Alert Loop

> *"The loop is the heartbeat of an agentic system. Engineer it well and your agent is reliable; engineer it poorly and it either never stops or stops too soon."*

**Book:** Agentic AI — From Foundations to Production  
**Part 5:** Building Production Agentic Systems  
**Chapter:** 19

## 🎯 Chapter Goal

By the end of this chapter you will be able to:

1. **Decompose** any agentic loop into 5 reusable primitives: Generator, Verifier, Terminator, StateManager, Escalator
2. **Implement** each primitive as a clean Python class with a single responsibility
3. **Build** the GAS Alert Loop that monitors continuous glucose readings
4. **Engineer** adaptive iteration budgets based on clinical urgency
5. **Detect** stagnation and escalate automatically when the loop is stuck
6. **Visualize** loop behavior across a hypoglycemia episode

## 📖 Theory Recap

### The Agentic Loop Problem

Every agentic system runs a loop: *observe → think → act → observe*. The challenge is knowing **when to stop**, **when to try harder**, and **when to hand off to a human**.

### The 5 Loop Primitives

| Primitive | Responsibility | Key Question |
|-----------|---------------|-------------|
| **Generator** | Produce the next response/action | *What should I do next?* |
| **Verifier** | Check if the response is valid/safe | *Is this response acceptable?* |
| **Terminator** | Decide when to stop looping | *Have we reached a satisfactory conclusion?* |
| **StateManager** | Update shared state after each iteration | *What changed? What do I remember?* |
| **Escalator** | Detect when human intervention is needed | *Is this beyond the agent's capability?* |

### Loop Engineering Principles

1. **Adaptive Budget**: Don't give every task the same iteration limit. Critical situations deserve more compute; routine checks need less.
2. **Stagnation Detection**: If the agent keeps producing the same output, it's stuck. Detect this and escalate.
3. **Graceful Degradation**: When the loop fails, fall back to a safe default — never leave the system in an unknown state.
4. **Observability**: Log every iteration with timestamp, state snapshot, and decision rationale.

### Glucose Alert Loop Architecture

```
CGM Reading → Generator → Verifier → StateManager
                  ↑              ↓
              Terminator ← Escalator
                  ↓
              [STOP / ESCALATE / CONTINUE]
```

### Adaptive Budget by Glucose Level

| Glucose Range | Classification | Max Iterations | Escalation |
|--------------|---------------|----------------|------------|
| 70–180 mg/dL | Normal | 1 | None |
| 54–70 or 180–250 | Borderline | 3 | Auto-alert |
| <54 or >250 | Critical | 5 | Human escalation |
| <40 or >350 | Severe | 5 + immediate | Emergency |


## 🔨 Build It

### Setup

In [ ]:
import os
import time
import random
import hashlib
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Optional

# TEST_MODE: run without any external API keys
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"

print(f"TEST_MODE: {TEST_MODE}")
print("Loop Engineering notebook ready.")

### Step 1: The 5 Loop Primitives

Each primitive is a single-responsibility class. Together they form a complete loop engine.

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVE 1: Generator
# Produces the next recommendation given current state
# ─────────────────────────────────────────────

class Generator:
    """
    Generates a clinical recommendation based on current glucose state.
    
    In production this would call an LLM. In TEST_MODE it uses
    deterministic rule-based logic.
    """

    def generate(self, state: dict) -> str:
        """
        Generate a recommendation string from the current state.
        
        Parameters
        ----------
        state : dict
            Must contain 'glucose' (float) and optionally 'trend', 'iteration'.
        
        Returns
        -------
        str
            A clinical recommendation.
        """
        glucose = state.get("glucose", 100)
        trend = state.get("trend", "stable")
        iteration = state.get("iteration", 0)

        if TEST_MODE or not os.getenv("OPENAI_API_KEY"):
            return self._rule_based_generate(glucose, trend, iteration)
        else:
            return self._llm_generate(state)

    def _rule_based_generate(self, glucose: float, trend: str, iteration: int) -> str:
        """Deterministic rule-based recommendation (no API needed)."""
        if glucose < 40:
            return "EMERGENCY: Glucose critically low. Call 911. Administer glucagon immediately."
        elif glucose < 54:
            return "CRITICAL: Severe hypoglycemia. Consume 15g fast-acting carbs NOW. Recheck in 15 min."
        elif glucose < 70:
            if trend == "falling":
                return "WARNING: Glucose falling into hypoglycemia range. Consume 15g carbs. Monitor closely."
            else:
                return "CAUTION: Glucose below target. Consume 10g carbs. Recheck in 15 min."
        elif glucose <= 180:
            return "OK: Glucose within target range. Continue current management."
        elif glucose <= 250:
            return "CAUTION: Glucose above target. Check for missed bolus. Consider correction dose."
        elif glucose <= 350:
            return "WARNING: Significant hyperglycemia. Administer correction dose. Check for ketones."
        else:
            return "CRITICAL: Severe hyperglycemia. Check ketones immediately. Contact provider."

    def _llm_generate(self, state: dict) -> str:
        """LLM-based generation (requires API key)."""
        # In production: call OpenAI/Anthropic with clinical context
        return self._rule_based_generate(
            state.get("glucose", 100),
            state.get("trend", "stable"),
            state.get("iteration", 0)
        )


# Quick test
gen = Generator()
for glucose in [38, 55, 65, 120, 200, 300, 400]:
    rec = gen.generate({"glucose": glucose, "trend": "falling"})
    print(f"  Glucose {glucose:3d}: {rec[:60]}..." if len(rec) > 60 else f"  Glucose {glucose:3d}: {rec}")

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVE 2: Verifier
# Checks if a generated response is safe and valid
# ─────────────────────────────────────────────

class Verifier:
    """
    Validates clinical recommendations for safety and completeness.
    
    Applies guardrails to prevent dangerous recommendations from
    reaching the patient.
    """

    # Dangerous patterns that should never appear in recommendations
    BLOCKED_PATTERNS = [
        "ignore glucose",
        "no action needed" ,  # when glucose is critical
        "take 20 units",      # dangerously high insulin dose
        "take 15 units",
    ]

    def verify(self, state: dict, response: str) -> tuple:
        """
        Verify a recommendation against safety rules.
        
        Parameters
        ----------
        state : dict
            Current system state (must contain 'glucose').
        response : str
            The recommendation to verify.
        
        Returns
        -------
        tuple[bool, str]
            (is_valid, reason) — reason explains why invalid if is_valid=False.
        """
        glucose = state.get("glucose", 100)
        response_lower = response.lower()

        # Rule 1: Block dangerous patterns
        for pattern in self.BLOCKED_PATTERNS:
            if pattern in response_lower:
                return False, f"Blocked pattern detected: '{pattern}'"

        # Rule 2: Critical glucose must have urgent language
        if glucose < 54:
            urgent_words = ["critical", "emergency", "immediately", "now", "severe"]
            if not any(w in response_lower for w in urgent_words):
                return False, "Critical glucose requires urgent language in recommendation"

        # Rule 3: Response must not be empty
        if not response or len(response.strip()) < 10:
            return False, "Response too short or empty"

        # Rule 4: Response must not exceed reasonable length
        if len(response) > 500:
            return False, "Response too long — may confuse patient"

        return True, "OK"


# Test the verifier
verifier = Verifier()
test_cases = [
    ({"glucose": 45}, "CRITICAL: Severe hypoglycemia. Consume 15g carbs NOW."),
    ({"glucose": 45}, "No action needed, glucose looks fine."),  # should fail
    ({"glucose": 120}, "OK: Glucose within target range."),
    ({"glucose": 45}, "Low."),  # too short
]

print("Verifier tests:")
for state, response in test_cases:
    valid, reason = verifier.verify(state, response)
    status = "✅ VALID" if valid else "❌ INVALID"
    print(f"  {status} | glucose={state['glucose']} | {reason}")
    print(f"    Response: {response[:60]}")

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVE 3: Terminator
# Decides when the loop has done enough work
# ─────────────────────────────────────────────

class Terminator:
    """
    Determines when the alert loop should stop iterating.
    
    Uses adaptive budgets: critical situations get more iterations,
    routine checks terminate quickly.
    """

    def get_budget(self, glucose: float) -> int:
        """
        Return the maximum iteration budget for a given glucose level.
        
        Adaptive budget strategy:
        - Normal (70-180): 1 iteration — quick check, no need to deliberate
        - Borderline (54-70 or 180-250): 3 iterations — verify and refine
        - Critical (<54 or >250): 5 iterations — thorough analysis
        """
        if 70 <= glucose <= 180:
            return 1   # Normal: single pass
        elif (54 <= glucose < 70) or (180 < glucose <= 250):
            return 3   # Borderline: verify and refine
        else:
            return 5   # Critical: thorough analysis

    def should_terminate(self, state: dict, iteration: int) -> bool:
        """
        Check if the loop should terminate.
        
        Parameters
        ----------
        state : dict
            Current state (must contain 'glucose', optionally 'verified').
        iteration : int
            Current iteration number (0-indexed).
        
        Returns
        -------
        bool
            True if the loop should stop.
        """
        glucose = state.get("glucose", 100)
        budget = self.get_budget(glucose)

        # Always terminate if budget exceeded
        if iteration >= budget:
            return True

        # Terminate early if we have a verified response and glucose is normal
        if state.get("verified", False) and 70 <= glucose <= 180:
            return True

        return False


# Demonstrate adaptive budgets
terminator = Terminator()
print("Adaptive iteration budgets:")
for glucose in [38, 55, 65, 120, 200, 260, 350]:
    budget = terminator.get_budget(glucose)
    label = "Normal" if 70 <= glucose <= 180 else ("Borderline" if (54 <= glucose < 70) or (180 < glucose <= 250) else "Critical")
    print(f"  Glucose {glucose:3d} mg/dL ({label:10s}): max {budget} iteration(s)")

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVE 4: StateManager
# Updates shared state after each iteration
# ─────────────────────────────────────────────

class StateManager:
    """
    Manages the evolving state of the alert loop.
    
    Tracks iteration history, verification results, and
    recommendation changes over time.
    """

    def update(self, state: dict, response: str, verification: str) -> dict:
        """
        Update state after one loop iteration.
        
        Parameters
        ----------
        state : dict
            Current state to update (mutated in place and returned).
        response : str
            The recommendation generated this iteration.
        verification : str
            Verification result: "OK" or an error message.
        
        Returns
        -------
        dict
            Updated state.
        """
        # Initialize history if needed
        if "history" not in state:
            state["history"] = []

        # Record this iteration
        state["history"].append({
            "iteration": state.get("iteration", 0),
            "response": response,
            "verification": verification,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "glucose": state.get("glucose"),
        })

        # Update current recommendation
        state["last_response"] = response
        state["verified"] = (verification == "OK")
        state["iteration"] = state.get("iteration", 0) + 1

        # Track consecutive identical responses (for stagnation detection)
        recent = [h["response"] for h in state["history"][-3:]]
        state["stagnant"] = len(recent) >= 3 and len(set(recent)) == 1

        return state

    def get_recommendation_hash(self, response: str) -> str:
        """Hash a response for stagnation detection."""
        return hashlib.md5(response.strip().lower().encode()).hexdigest()[:8]


# Quick test
sm = StateManager()
state = {"glucose": 55, "trend": "falling", "iteration": 0}
state = sm.update(state, "CAUTION: Consume 15g carbs.", "OK")
state = sm.update(state, "CAUTION: Consume 15g carbs.", "OK")
state = sm.update(state, "CAUTION: Consume 15g carbs.", "OK")
print(f"After 3 identical responses, stagnant={state['stagnant']}")
print(f"History length: {len(state['history'])}")

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVE 5: Escalator
# Detects when human intervention is needed
# ─────────────────────────────────────────────

class Escalator:
    """
    Detects conditions requiring human escalation and handles the escalation.
    
    Escalation triggers:
    1. Glucose critically low (<54) or high (>300)
    2. Loop stagnation (3+ identical recommendations)
    3. Verification failures on critical readings
    4. Rapid glucose decline (>15 mg/dL in last 2 readings)
    """

    def __init__(self):
        self.escalation_log = []

    def should_escalate(self, state: dict) -> bool:
        """
        Determine if the current state requires human escalation.
        
        Parameters
        ----------
        state : dict
            Current loop state.
        
        Returns
        -------
        bool
            True if escalation is needed.
        """
        glucose = state.get("glucose", 100)

        # Trigger 1: Critical glucose levels
        if glucose < 54 or glucose > 300:
            return True

        # Trigger 2: Stagnation detected
        if state.get("stagnant", False):
            return True

        # Trigger 3: Rapid decline
        history = state.get("glucose_history", [])
        if len(history) >= 2:
            recent_drop = history[-2] - history[-1]
            if recent_drop > 15:  # >15 mg/dL drop between readings
                return True

        return False

    def escalate(self, state: dict) -> dict:
        """
        Perform escalation: log the event and update state.
        
        Parameters
        ----------
        state : dict
            Current loop state.
        
        Returns
        -------
        dict
            Updated state with escalation info.
        """
        glucose = state.get("glucose", 100)
        reason = self._determine_reason(state)

        escalation_event = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "glucose": glucose,
            "reason": reason,
            "state_snapshot": {
                "iteration": state.get("iteration", 0),
                "last_response": state.get("last_response", ""),
                "stagnant": state.get("stagnant", False),
            }
        }

        self.escalation_log.append(escalation_event)
        state["escalated"] = True
        state["escalation_reason"] = reason
        state["escalation_time"] = escalation_event["timestamp"]

        print(f"  🚨 ESCALATION: {reason} (glucose={glucose})")
        return state

    def _determine_reason(self, state: dict) -> str:
        """Determine the primary escalation reason."""
        glucose = state.get("glucose", 100)
        if glucose < 40:
            return "EMERGENCY: Glucose critically low (<40 mg/dL)"
        elif glucose < 54:
            return "CRITICAL: Severe hypoglycemia (<54 mg/dL)"
        elif glucose > 350:
            return "CRITICAL: Severe hyperglycemia (>350 mg/dL)"
        elif glucose > 300:
            return "WARNING: Significant hyperglycemia (>300 mg/dL)"
        elif state.get("stagnant", False):
            return "STAGNATION: Loop stuck — same recommendation 3+ times"
        else:
            return "RAPID_DECLINE: Glucose dropping rapidly"


# Test escalator
escalator = Escalator()
test_states = [
    {"glucose": 120, "stagnant": False},
    {"glucose": 45, "stagnant": False},
    {"glucose": 120, "stagnant": True},
    {"glucose": 65, "glucose_history": [90, 75, 55], "stagnant": False},
]
print("Escalation decisions:")
for s in test_states:
    should = escalator.should_escalate(s)
    print(f"  glucose={s['glucose']:3d}, stagnant={s.get('stagnant',False)} → escalate={should}")

### Step 2: The GAS Alert Loop

Now we assemble the 5 primitives into a complete loop engine.

In [ ]:
# ─────────────────────────────────────────────
# GAS Alert Loop
# Assembles all 5 primitives into a complete loop
# ─────────────────────────────────────────────

@dataclass
class LoopTrace:
    """Records the complete execution trace of one loop run."""
    glucose: float
    iterations: int
    final_recommendation: str
    escalated: bool
    escalation_reason: str
    duration_ms: float
    history: list = field(default_factory=list)


class GASAlertLoop:
    """
    The Guardian Angel System Alert Loop.
    
    Orchestrates the 5 loop primitives to produce safe, verified
    clinical recommendations for each glucose reading.
    
    This IS the GAS Monitor Agent's core loop.
    """

    def __init__(self):
        self.generator = Generator()
        self.verifier = Verifier()
        self.terminator = Terminator()
        self.state_manager = StateManager()
        self.escalator = Escalator()
        self.run_log = []

    def run(self, glucose: float, trend: str = "stable",
            glucose_history: list = None) -> LoopTrace:
        """
        Run the alert loop for a single glucose reading.
        
        Parameters
        ----------
        glucose : float
            Current glucose reading in mg/dL.
        trend : str
            Glucose trend: 'rising', 'stable', or 'falling'.
        glucose_history : list[float]
            Recent glucose readings for trend analysis.
        
        Returns
        -------
        LoopTrace
            Complete execution trace.
        """
        start_time = time.time()

        # Initialize state
        state = {
            "glucose": glucose,
            "trend": trend,
            "glucose_history": glucose_history or [glucose],
            "iteration": 0,
            "verified": False,
            "stagnant": False,
            "escalated": False,
            "history": [],
        }

        budget = self.terminator.get_budget(glucose)
        final_recommendation = ""

        print(f"\n{'='*60}")
        print(f"GAS Alert Loop | glucose={glucose} mg/dL | trend={trend} | budget={budget}")
        print(f"{'='*60}")

        for i in range(budget + 1):  # +1 to allow one final escalation check
            # Check termination
            if self.terminator.should_terminate(state, i):
                print(f"  [iter {i}] Terminator: STOP (budget={budget}, iteration={i})")
                break

            # Generate recommendation
            response = self.generator.generate(state)
            print(f"  [iter {i}] Generator: {response[:70]}{'...' if len(response)>70 else ''}")

            # Verify recommendation
            is_valid, reason = self.verifier.verify(state, response)
            print(f"  [iter {i}] Verifier: {'✅ valid' if is_valid else f'❌ invalid — {reason}'}")

            # Update state
            state = self.state_manager.update(state, response, reason if not is_valid else "OK")

            if is_valid:
                final_recommendation = response

            # Check escalation
            if self.escalator.should_escalate(state):
                state = self.escalator.escalate(state)
                break

        duration_ms = (time.time() - start_time) * 1000

        trace = LoopTrace(
            glucose=glucose,
            iterations=state.get("iteration", 0),
            final_recommendation=final_recommendation or state.get("last_response", ""),
            escalated=state.get("escalated", False),
            escalation_reason=state.get("escalation_reason", ""),
            duration_ms=round(duration_ms, 2),
            history=state.get("history", []),
        )

        self.run_log.append(trace)
        print(f"\n  ✅ Final: {trace.final_recommendation[:80]}")
        print(f"  ⏱  {trace.iterations} iteration(s) in {trace.duration_ms:.1f}ms")
        if trace.escalated:
            print(f"  🚨 Escalated: {trace.escalation_reason}")

        return trace


print("GASAlertLoop defined.")

### Step 3: Run the Hypoglycemia Episode Scenario

Simulate a patient's glucose dropping from 95 → 38 mg/dL over 6 readings.

In [ ]:
# Hypoglycemia episode: glucose readings over ~30 minutes
# Each reading is 5 minutes apart
glucose_readings = [95, 82, 68, 54, 42, 38]
trends = ["stable", "falling", "falling", "falling", "falling", "falling"]

loop = GASAlertLoop()
traces = []

print("\n🏥 HYPOGLYCEMIA EPISODE SIMULATION")
print("Patient: Alice (T1D), nocturnal hypoglycemia scenario")
print("Readings taken every 5 minutes\n")

for i, (glucose, trend) in enumerate(zip(glucose_readings, trends)):
    history = glucose_readings[:i+1]
    trace = loop.run(glucose, trend, history)
    traces.append(trace)

In [ ]:
# Summary table
print("\n" + "="*80)
print("EPISODE SUMMARY")
print("="*80)
print(f"{'Reading':>8} {'Glucose':>8} {'Iters':>6} {'Escalated':>10} {'Recommendation':<50}")
print("-"*80)

for i, trace in enumerate(traces):
    esc = "🚨 YES" if trace.escalated else "  no"
    rec = trace.final_recommendation[:48] + ".." if len(trace.final_recommendation) > 50 else trace.final_recommendation
    print(f"  {i+1:>5}   {trace.glucose:>6.0f}   {trace.iterations:>5}   {esc:>9}   {rec}")

print("="*80)
print(f"Total escalations: {sum(1 for t in traces if t.escalated)}")
print(f"Total iterations:  {sum(t.iterations for t in traces)}")
print(f"Avg latency:       {sum(t.duration_ms for t in traces)/len(traces):.1f}ms")

### Step 4: Stagnation Detection Demo

Show what happens when the loop gets stuck producing the same recommendation.

In [ ]:
# Stagnation demo: borderline glucose that keeps getting the same recommendation
print("\n🔄 STAGNATION DETECTION DEMO")
print("Scenario: Glucose stuck at 65 mg/dL (borderline), loop gets same answer 3x")
print()

# Manually simulate stagnation
sm = StateManager()
escalator = Escalator()

state = {"glucose": 65, "trend": "stable", "iteration": 0, "history": []}
same_response = "CAUTION: Glucose below target. Consume 10g carbs. Recheck in 15 min."

for i in range(4):
    state = sm.update(state, same_response, "OK")
    stagnant = state.get("stagnant", False)
    should_esc = escalator.should_escalate(state)
    print(f"  Iteration {i+1}: stagnant={stagnant}, should_escalate={should_esc}")
    if should_esc:
        state = escalator.escalate(state)
        print(f"  → Escalation triggered: {state['escalation_reason']}")
        break

### Step 5: Visualization — Loop Iterations vs Glucose Level

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

reading_nums = list(range(1, len(traces) + 1))
glucoses = [t.glucose for t in traces]
iterations = [t.iterations for t in traces]
escalated = [t.escalated for t in traces]

# ── Top panel: Glucose trace ──
ax1.axhspan(0, 54, alpha=0.15, color="red")
ax1.axhspan(54, 70, alpha=0.10, color="orange")
ax1.axhspan(70, 180, alpha=0.10, color="green")
ax1.axhline(70, color="orange", linestyle="--", linewidth=1, alpha=0.7)
ax1.axhline(54, color="red", linestyle=":", linewidth=1, alpha=0.7)

ax1.plot(reading_nums, glucoses, "b-o", linewidth=2, markersize=8, label="Glucose")

# Mark escalation events with red dots
esc_x = [reading_nums[i] for i, e in enumerate(escalated) if e]
esc_y = [glucoses[i] for i, e in enumerate(escalated) if e]
if esc_x:
    ax1.scatter(esc_x, esc_y, color="red", s=200, zorder=5, label="Escalation", marker="*")

ax1.set_ylabel("Glucose (mg/dL)", fontsize=11)
ax1.set_title("GAS Alert Loop: Hypoglycemia Episode", fontsize=13, fontweight="bold")
ax1.legend(loc="upper right")
ax1.set_ylim(20, 110)
ax1.grid(axis="y", alpha=0.3)

# Add zone labels
ax1.text(0.02, 45, "Severe Hypo", transform=ax1.get_yaxis_transform(), color="red", fontsize=8, alpha=0.7)
ax1.text(0.02, 60, "Hypo", transform=ax1.get_yaxis_transform(), color="darkorange", fontsize=8, alpha=0.7)
ax1.text(0.02, 80, "Target", transform=ax1.get_yaxis_transform(), color="green", fontsize=8, alpha=0.7)

# ── Bottom panel: Iterations used ──
colors = ["red" if e else ("orange" if it > 1 else "steelblue") for it, e in zip(iterations, escalated)]
bars = ax2.bar(reading_nums, iterations, color=colors, alpha=0.8, edgecolor="white")

# Add budget line
budgets = [Terminator().get_budget(g) for g in glucoses]
ax2.step(reading_nums, budgets, where="mid", color="gray", linestyle="--", linewidth=1.5, label="Budget")

ax2.set_xlabel("Reading Number (5-min intervals)", fontsize=11)
ax2.set_ylabel("Loop Iterations", fontsize=11)
ax2.set_title("Iterations Used vs Budget", fontsize=11)
ax2.set_xticks(reading_nums)
ax2.set_xticklabels([f"#{i}\n{g:.0f}" for i, g in zip(reading_nums, glucoses)])
ax2.set_ylim(0, 7)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

# Legend patches
legend_patches = [
    mpatches.Patch(color="steelblue", alpha=0.8, label="Normal (1 iter)"),
    mpatches.Patch(color="orange", alpha=0.8, label="Borderline (3 iter)"),
    mpatches.Patch(color="red", alpha=0.8, label="Critical/Escalated"),
]
ax2.legend(handles=legend_patches, loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig("loop_iterations_vs_glucose.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved to loop_iterations_vs_glucose.png")

## 🏥 GAS Connection

### This IS the GAS Monitor Agent's Core Loop

The `GASAlertLoop` you just built is not a toy — it's the actual architecture of the GAS Monitor Agent:

```
GAS Monitor Agent
├── Generator      → LLM call with clinical context
├── Verifier       → Safety guardrails (dose limits, urgency checks)
├── Terminator     → Adaptive budget by glucose severity
├── StateManager   → Tracks history, stagnation, verification
└── Escalator      → Triggers caregiver/emergency alerts
```

### How Each Primitive Maps to GAS

| Primitive | GAS Role | Production Implementation |
|-----------|----------|---------------------------|
| **Generator** | Calls GPT-4 with patient context + guidelines | `llm_client.complete(messages)` |
| **Verifier** | Checks dose safety, urgency language | Rule engine + secondary LLM |
| **Terminator** | Adaptive budget by glucose severity | Configurable per patient |
| **StateManager** | Updates GASState dataclass | Redis-backed state store |
| **Escalator** | Sends SMS/push to caregiver | Twilio + Firebase |

### The Adaptive Budget is a Patient Safety Feature

By giving critical readings 5x more compute than routine checks, GAS ensures:
- **Speed** for normal readings (no unnecessary latency)
- **Thoroughness** for dangerous readings (multiple verification passes)
- **Escalation** when the loop can't resolve the situation alone

This is **loop engineering as a safety mechanism**, not just an optimization.

## ✅ Checkpoint

### What You Built

| Component | Lines of Code | Key Insight |
|-----------|--------------|-------------|
| `Generator` | ~40 | Single responsibility: produce next action |
| `Verifier` | ~35 | Safety guardrails as first-class citizens |
| `Terminator` | ~30 | Adaptive budget = safety + efficiency |
| `StateManager` | ~35 | Stagnation detection via history tracking |
| `Escalator` | ~50 | Human-in-the-loop as a loop primitive |
| `GASAlertLoop` | ~60 | Orchestration of all 5 primitives |

### Key Takeaways

1. **Decompose loops into 5 primitives** — each with a single responsibility
2. **Adaptive budgets** prevent both under-computation (missing critical events) and over-computation (unnecessary latency)
3. **Stagnation detection** is essential — loops that repeat the same answer are stuck, not converging
4. **Escalation is a loop primitive**, not an afterthought — design it in from the start
5. **Observability** (the `LoopTrace`) makes debugging production loops tractable

### Self-Check Questions

1. What happens if the Verifier rejects every response the Generator produces?
2. How would you modify the Terminator to account for time-of-day (e.g., stricter at night)?
3. What's the difference between stagnation (same answer) and convergence (correct answer)?
4. How would you add a "confidence score" to the Generator output?

### Next Chapter

**Chapter 20: Graph Engineering** — Model complex agent workflows as directed graphs, enabling branching, parallel execution, and cycle detection.